# From Voice to Vision — 1. Data Acquisition and Exploration

This notebook downloads the RAVDESS speech corpus, decodes the emotion labels from the
standardised file names and performs a first exploratory analysis of the data.

RAVDESS contains 1440 utterances performed by 24 professional actors across eight emotional
states. Labels, actor identity and emotional intensity are all encoded in the file name, so no
external annotation file is required. The partitioning is *speaker-independent*: the actors used
for validation and testing never appear in training, so that measured accuracy reflects
generalisation to unseen voices rather than memorisation of speaker identity.

In [ ]:
# Clone the project repository and install the audio dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

In [ ]:
# Download and extract the corpus (~200 MB)
from src import config, data_loader
data_loader.download_ravdess()

In [ ]:
# Build the dataset index: one row of metadata per clip
import pandas as pd
df = data_loader.build_index()
print("Total clips:", len(df))
df.head()

In [ ]:
# Verify the speaker-independent partitioning and the class balance
print("Clips per split:")
print(df["split"].value_counts(), "\n")
print("Actors per split (must not overlap):")
for s in ["train", "val", "test"]:
    print(f"  {s:5s}:", sorted(df[df.split == s].actor.unique()))
print("\nClips per emotion:")
print(df["emotion"].value_counts())

The corpus is balanced at 192 clips per emotion, with the exception of *neutral* (96 clips),
which lacks the "strong" intensity level. This mild imbalance is compensated later through
class-balanced loss weights rather than resampling.

In [ ]:
# Emotion distribution across the three splits
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 4))
sns.countplot(data=df, x="emotion", order=config.EMOTIONS, hue="split")
plt.title("Emotion distribution per split (RAVDESS)")
plt.xticks(rotation=30); plt.ylabel("number of clips"); plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "00_emotion_distribution.png", dpi=150)
plt.show()

In [ ]:
# Example clip: waveform and log-Mel spectrogram
import librosa, librosa.display
import numpy as np
from IPython.display import Audio, display

sample = df[df.emotion == "angry"].iloc[0]
y, sr = librosa.load(sample.path, sr=config.SAMPLE_RATE)
print(f"Emotion: {sample.emotion} | actor: {sample.actor} ({sample.gender}) "
      f"| duration: {len(y)/sr:.2f} s")
display(Audio(y, rate=sr))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
librosa.display.waveshow(y, sr=sr, ax=ax[0])
ax[0].set_title("Waveform"); ax[0].set_xlabel("Time (s)"); ax[0].set_ylabel("Amplitude")

S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=config.N_MELS,
                                   n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
S_db = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_db, sr=sr, hop_length=config.HOP_LENGTH,
                               x_axis="time", y_axis="mel", ax=ax[1])
ax[1].set_title("log-Mel spectrogram"); ax[1].set_xlabel("Time (s)")
fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "00_sample_angry.png", dpi=150)
plt.show()